# Local Inference With Ollama

<div class="alert alert-success">  
    
### Learning Objectives 
    
* Run modern language models locally on Mac (Intel/Apple Silicon) or PC with Ollama
* Work with efficient models optimized for CPU/Apple Silicon
* Understand practical limits of local inference
* Master text generation without cloud dependencies
* Implement efficient processing for research tasks

</div>

This notebook shows you how to run open-source LLMs **locally on your laptop** for text analysis tasks like sentiment analysis and classification using **Ollama**.

## What is Ollama?

**Ollama** is the easiest way to run LLMs locally. It is a lightweight system that lets you download and run open-weight language models entirely on your laptop (Mac, Windows, or Linux) with almost no setup. It handles downloading, running, and managing models with simple commands.

### Key Advantages of Ollama:

1. **Dead Simple**: `ollama pull qwen2.5:3b` and you're ready
2. **Model Library**: Browse and download models with one command
3. **Automatic Updates**: Models stay current
4. **Clean API**: Python interface is intuitive
5. **Built-in Server**: REST API included

### Installation:

1. Download Ollama from [ollama.com](https://ollama.com)
2. Install and start it
3. That's it!

### Popular Models on Ollama:

- `qwen2.5:3b` - Fast, efficient (recommended for this tutorial)
- `qwen2.5:7b` - Better quality, slower
- `llama3.2:3b` - Meta's small model
- `phi3:mini` - Microsoft's efficient model
- `mistral:7b` - Popular general-purpose model

View all models: [ollama.com/library](https://ollama.com/library)

## Setup

Install the required libraries:

In [1]:
# Install ollama Python client and dependencies
%pip install ollama 
# %pip install pydantic instructor

Note: you may need to restart the kernel to use updated packages.


## Download a Model

### Step 1: Install and Start Ollama

Ollama runs as a local server on your machine (like a mini API), and the Python client sends requests to it just like you'd call an API, except everything stays on your laptop.

1. **Download Ollama**: Visit [ollama.com](https://ollama.com/download) and download the installer for your OS
2. **Install**: Run the installer (it's quick!)
3. **Verify it's running**: 
   - **Mac/Linux**: Ollama starts automatically after installation
   - **Windows**: Look for Ollama in your system tray
   - You can also check by running `ollama --version` in your terminal

### Step 2: Pull a Model

Once Ollama is running, pull the model using Python:

In [3]:
import ollama

# Pull the model (this will download it if not already present)
# This is a one-time download
ollama.pull('qwen2.5:3b')

ProgressResponse(status='success', completed=None, total=None, digest=None)

**Available model sizes:**
- `qwen2.5:0.5b` - Ultra-lightweight (~400MB)
- `qwen2.5:1.5b` - Small and fast (~1GB)
- `qwen2.5:3b` - Recommended balance (~2GB)
- `qwen2.5:7b` - Better quality (~4.7GB)
- `qwen2.5:14b` - High quality (~9GB)

## Simple Text Generation

Let's start with basic inference:

In [9]:
import time

prompt = "Write a short poem about artificial intelligence"

# Time the generation
start = time.time()

response = ollama.chat(
    model='qwen2.5:3b',
    messages=[
        {'role': 'user', 'content': prompt}
    ],
    options={
        'temperature': 0.1,
        'num_predict': 100  # Match MLX's max_tokens
    }
)

elapsed = time.time() - start

# Rough token estimate (Ollama doesn't expose exact count)
text = response['message']['content']
estimated_tokens = len(text.split()) * 1.3
tokens_per_second = estimated_tokens / elapsed

print(f"Time: {elapsed:.2f}s")
print(f"Estimated tokens: {int(estimated_tokens)}")
print(f"Speed: {tokens_per_second:.1f} tokens/second\n")
print(f"Response: {text}")

Time: 1.86s
Estimated tokens: 61
Speed: 32.9 tokens/second

Response: In circuits and in code,
Lies the spirit of our love.
From silicon whispers soft,
To guide us through the night.

We build you with great care,
Yet fear your power's sway.
A dance between creation and control,
As we learn to tread this new, vast floor.


**Problem:** The output is unstructured text - hard to parse programmatically!

## Structured Output with JSON Schema

Ollama supports structured outputs via the `format` parameter:

In [10]:
from pydantic import BaseModel
from typing import Literal
import json

# Define the structure we want
class SentimentAnalysis(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float  # 0.0 to 1.0
    reasoning: str

# Example text
text = "This product exceeded my expectations! The quality is amazing and shipping was fast."

# Get structured output
response = ollama.chat(
    model='qwen2.5:3b',
    messages=[
        {'role': 'system', 'content': 'You are a sentiment analysis expert. Always respond with valid JSON.'},
        {'role': 'user', 'content': f'Analyze the sentiment of this text: {text}'}
    ],
    format=SentimentAnalysis.model_json_schema()  # Force JSON schema
)

# Parse the JSON response
result = SentimentAnalysis.model_validate_json(response['message']['content'])

print(f"Sentiment: {result.sentiment}")
print(f"Confidence: {result.confidence}")
print(f"Reasoning: {result.reasoning}")

Sentiment: positive
Confidence: 0.95
Reasoning: The text contains phrases such as 'exceeded my expectations', 'quality is amazing', and 'shipping was fast' which are all indicative of positive sentiment.


## Multi-Class Text Classification

Let's classify social media posts into topics:

In [15]:
from typing import List

class TopicClassification(BaseModel):
    primary_topic: Literal["technology", "politics", "sports", "entertainment", "science", "other"]
    all_topics: List[str]
    confidence: float
    summary: str

post = """Just watched the new AI announcement from OpenAI. 
The tech industry is moving so fast! Can't wait to try GPT-5."""

response = ollama.chat(
    model='qwen2.5:3b',
    messages=[
        {'role': 'system', 'content': 'You classify social media posts by topic.'},
        {'role': 'user', 'content': f'Classify this post: {post}'}
    ],
    format=TopicClassification.model_json_schema()
)

# Parse the response
result = TopicClassification.model_validate_json(response['message']['content'])

print(f"Primary Topic: {result.primary_topic}")
print(f"All Topics: {result.all_topics}")
print(f"Confidence: {result.confidence}")
print(f"Summary: {result.summary}")

Primary Topic: technology
All Topics: ['ai', 'tech_industry']
Confidence: 0.95
Summary: User expresses excitement about a recent AI announcement and looks forward to trying new technology like GPT-5.


## Batch Processing Your Data

In [17]:
import pandas as pd
from tqdm import tqdm

# Example dataset
texts = [
    "I love this movie! Best film of the year.",
    "Terrible service. Will never come back.",
    "It was okay, nothing special.",
    "Absolutely amazing experience from start to finish!",
    "Not impressed. Expected better quality."
]

def analyze_sentiment(text):
    """Analyze sentiment for a single text"""
    response = ollama.chat(
        model='qwen2.5:3b',
        messages=[
            {'role': 'system', 'content': 'You analyze sentiment.'},
            {'role': 'user', 'content': f'Analyze: {text}'}
        ],
        format=SentimentAnalysis.model_json_schema()
    )
    return SentimentAnalysis.model_validate_json(response['message']['content'])

In [18]:
# Process all texts
results = []
for text in tqdm(texts, desc="Analyzing"):
    result = analyze_sentiment(text)
    results.append({
        'text': text,
        'sentiment': result.sentiment,
        'confidence': result.confidence,
        'reasoning': result.reasoning
    })

# Create DataFrame
df = pd.DataFrame(results)
df

Analyzing: 100%|██████████| 5/5 [00:11<00:00,  2.28s/it]


,text,sentiment,confidence,reasoning
0,I love this movie! Best film of the year.,positive,1.00,The statement contains clear positive indicato...
1,Terrible service. Will never come back.,negative,1.00,The statement 'Terrible service' directly expr...
2,"It was okay, nothing special.",negative,95.00,"The phrase 'It was okay, nothing special' indi..."
3,Absolutely amazing experience from start to fi...,positive,1.00,The phrase 'Absolutely amazing' indicates stro...
4,Not impressed. Expected better quality.,negative,0.95,The phrase 'Not impressed' and the statement '...


## Using Your Own Data

In [ ]:
# Load your data
# df = pd.read_csv('your_data.csv')

# Define your custom classification schema
class CustomClassification(BaseModel):
    category: Literal["category1", "category2", "category3"]
    confidence: float
    key_phrases: List[str]

# Apply to your dataframe
# df['classification'] = df['your_text_column'].progress_apply(classify_text)

## Checking Model Uncertainty

In [30]:
import numpy as np
import matplotlib.pyplot as plt

# Prompt for next token prediction
prompt = "One word defining Americans:"

# Get logprobs with generate (not chat - chat doesn't support logprobs yet)
response = ollama.generate(
    model='qwen2.5:3b',
    prompt=prompt,
    options={
        'num_predict': 3,  # get 3 tokens
        'temperature': 0.0
    },
    logprobs=True,
    top_logprobs=5  # Get top 5 candidates
)

# Extract token probabilities
top_logprobs = response['logprobs'][0]['top_logprobs']

# Convert to readable format
tokens = [item['token'] for item in top_logprobs]
logprobs = [item['logprob'] for item in top_logprobs]
probs = [np.exp(lp) * 100 for lp in logprobs]  # Convert to percentages

# Visualize all 3 tokens
for i, token_info in enumerate(response['logprobs']):
    print(f"\nToken {i+1}: '{token_info['token']}'")
    for alt in token_info['top_logprobs'][:3]:
        prob = np.exp(alt['logprob']) * 100
        print(f"  {alt['token']}: {prob:.1f}%")


Token 1: 'Individual'
  Individual: 47.5%
  Ad: 15.3%
  E: 7.0%

Token 2: 'ism'
  ism: 98.5%
  istic: 0.7%
  ist: 0.7%


## Managing Models

Useful Ollama commands:

In [31]:
ollama.list()

ListResponse(models=[Model(model='qwen2.5:3b', modified_at=datetime.datetime(2025, 11, 18, 8, 11, 12, 126329, tzinfo=TzInfo(-28800)), digest='357c53fb659c5076de1d65ccb0b397446227b71a42be9d1603d46168015c9e4b', size=1929912432, details=ModelDetails(parent_model='', format='gguf', family='qwen2', families=['qwen2'], parameter_size='3.1B', quantization_level='Q4_K_M'))])

In [32]:
# List installed models
models = ollama.list()
for model in models['models']:
    print(f"{model['model']} - {model['size'] / 1e9:.1f}GB")

# Delete a model to free space
# ollama.delete('qwen2.5:7b')

# Pull a different model
# ollama.pull('llama3.2:3b')

qwen2.5:3b - 1.9GB


## Tips for Better Results

1. **System prompts matter**: Be specific about the task
2. **Temperature**: Use 0 for consistent results, 0.7+ for variety
3. **Check consistency**: Run ambiguous texts multiple times
4. **Try different models**: Larger models = better quality but slower

## Resources:

- [Ollama Website](https://ollama.com)
- [Ollama Model Library](https://ollama.com/library)
- [Ollama Python Docs](https://github.com/ollama/ollama-python)
- [Instructor docs](https://python.useinstructor.com/)
- [Pydantic docs](https://docs.pydantic.dev/)